In [3]:
import requests
import pandas as pd
from tconnectsync.sync import basal, bolus, cgm, pump_events

In [8]:
from collections.abc import Mapping
import datetime
import tconnectsync as tcs
import pandas as pd
import os
from dotenv import load_dotenv
import logging
import openpyxl

tcs.util.cli.enable_logging()
api = tcs.util.cli.get_api()    

In [12]:
load_dotenv()
TCONNECT_EMAIL = os.getenv('TCONNECT_EMAIL')
TCONNECT_PASSWORD = os.getenv('TCONNECT_PASSWORD')
PUMP_SERIAL_NUMBER = os.getenv('PUMP_SERIAL_NUMBER')

"""A wrapper for the three different t:connect API types."""
class TConnectApi:
    email = None
    password = None

    def __init__(self, email, password):
        self.email = email
        self.password = password
        self._ciq = None

    @property
    def controliq(self):
        if self._ciq and not self._ciq.needs_relogin():
            return self._ciq

        logger.debug("Instantiating new ControlIQApi")

        self._ciq = ControlIQApi(self.email, self.password)
        return self._ciq

tconnect = TConnectApi(TCONNECT_EMAIL, TCONNECT_PASSWORD)

In [14]:


def dict_and_list_obj_to_dataframe(data):
    """
    Convert an object containing nested dictionaries and lists to a Pandas DataFrame and write it to a CSV file.

    Parameters:
        data (dict or list): The object to convert to a DataFrame.
        csv_filename (str): The filename to write the resulting CSV file to.
        
    Returns:
        None
    """
    # Convert the object to a dictionary
    if isinstance(data, list):
        data = {'data': data}

    def flatten_dict(d, parent_key='', sep='_'):
        """
        Helper function to flatten nested dictionaries.
        """
        items = []
        for k, v in d.items():
            new_key = f"{parent_key}{sep}{k}" if parent_key else k
            if isinstance(v, dict):
                items.extend(flatten_dict(v, new_key, sep=sep).items())
            elif isinstance(v, list):
                for i, item in enumerate(v):
                    items.extend(flatten_dict(
                        item, f"{new_key}{sep}{i}", sep=sep).items())
            else:
                items.append((new_key, v))
        return dict(items)

    # Flatten the nested dictionaries and convert to a DataFrame
    flat_dict = flatten_dict(data)
    df = pd.DataFrame.from_dict(flat_dict, orient='index')

    return (df)


def to_dataframe(data):
    """
    Convert a dictionary with nested dictionaries and lists to a Pandas DataFrame and write it to a CSV file.

    Parameters:
        data (dict): The dictionary to convert to a DataFrame.
        csv_filename (str): The filename to write the resulting CSV file to.
        
    Returns:
        None
    """
    # Extract the event list from the data dictionary
    events = data['event']

    # Create a list to hold the flattened event dictionaries
    flat_events = []

    # Loop over each event dictionary and flatten it
    for event in events:
        flat_event = {}
        for k, v in event.items():
            if isinstance(v, dict):
                for kk, vv in v.items():
                    flat_event[f"{k}_{kk}"] = vv
            else:
                flat_event[k] = v
        flat_events.append(flat_event)

    # Convert the flattened event dictionaries to a DataFrame
    df = pd.DataFrame(flat_events)

    # Write the DataFrame to a CSV file
    return (df)


def flatten_dict_to_dfs(dct):
    dfs = {}
    for key, value in dct.items():
        if isinstance(value, Mapping):
            subdfs = flatten_dict_to_dfs(value)
            for subkey, subdf in subdfs.items():
                dfs[f"{key}_{subkey}"] = subdf
        elif isinstance(value, list):
            if all(isinstance(item, Mapping) for item in value):
                subdfs = flatten_dict_to_dfs(
                    {f"{key}_{i}": item for i, item in enumerate(value)})
                for subkey, subdf in subdfs.items():
                    dfs[subkey] = subdf
            else:
                df = pd.DataFrame(value)
                if len(df.columns) > 0:
                    dfs[key] = df
                else:
                    dfs[key] = pd.DataFrame(columns=["value"])
        else:
            dfs[key] = pd.DataFrame({"value": [value]})

        # add duration-related columns
        if key.endswith("_duration"):
            df = dfs[key]
            df["hours_passed"] = df["value"] / 3600
            df["days_passed"] = df["hours_passed"] / 24
            dfs[key] = df

    return dfs

def write_control_iq_data():
    project_dir = os.getcwd()
    data_dir = os.path.join(project_dir, "data")
    if not os.path.exists(data_dir):
        os.makedirs(data_dir)

    events_dir = os.path.join(data_dir, "cgm_and_bolus")
    if not os.path.exists(events_dir):
        os.makedirs(events_dir)

    basal_dir = os.path.join(data_dir, "basal")
    if not os.path.exists(basal_dir):
        os.makedirs(basal_dir)

    # Get current date
    current_date = datetime.datetime.now().date()

    # Iterate over the months from Jan 1, 2022 up to the current month
    start_date = datetime.date(2023, 6, 1)
    end_date = datetime.date(
        current_date.year, current_date.month, 1) + pd.offsets.MonthEnd(0)

    for month in pd.date_range(start=start_date, end=end_date, freq='MS'):
        # create the filename
        filename_events = os.path.join(
            events_dir, f"therapy_events_{month.strftime('%Y-%m')}.xlsx")
        filename_timeline = os.path.join(
            basal_dir, f"therapy_timeline_{month.strftime('%Y-%m')}.xlsx")

        # Get the therapy timeline for the current month
        therapy_events = api.controliq.therapy_events(month.strftime(
            '%Y-%m-%d'), (month + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d'))

        therapy_timeline = api.controliq.therapy_timeline(month.strftime(
            '%Y-%m-%d'), (month + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d'))

        # if all(not v for v in therapy_timeline['basal']['profileRates']):
        #     print("Did not find any values for this month.")
        #     continue
        # else:
        print(month)
        print(filename_events)
        print(filename_timeline)

        # # Convert the list-dictionary combined object to a DataFrame and write to CSV
        # for i in therapy_timeline.items():
        #     print(i)
        
        # Flatten the dictionary into a dictionary of dataframes
        dfs = flatten_dict_to_dfs(therapy_timeline)
        print(therapy_timeline)
        # for key, df in dfs.items():
        #     print(f"{key}:")
        #     print(df)

        # Write each dataframe to a separate Excel sheet
        with pd.ExcelWriter(filename_timeline) as writer:
            for key, df in dfs.items():
                # print(df)
                df.to_excel(writer, sheet_name=key, index=False)
        
        to_dataframe(therapy_events).to_excel(
            filename_events, index=True)
        # dict_and_list_obj_to_dataframe(therapy_timeline).to_excel(
        #     filename_timeline, index=True)
        
    print("Done!")


write_control_iq_data()


2023-06-01 00:00:00
/Users/ncamarda/Desktop/coding/bayesian-t1dm/data/cgm_and_bolus/therapy_events_2023-06.xlsx
/Users/ncamarda/Desktop/coding/bayesian-t1dm/data/basal/therapy_timeline_2023-06.xlsx
{'basal': {'profileRates': [{'y': 0.75, 'duration': 275, 'x': 1685602800}, {'y': 0.76, 'duration': 10780, 'x': 1685603075}, {'y': 0.78, 'duration': 10778, 'x': 1685613855}, {'y': 0.8, 'duration': 10779, 'x': 1685624633}, {'y': 0.795, 'duration': 10868, 'x': 1685635412}, {'y': 0.82, 'duration': 10660, 'x': 1685646280}, {'y': 0.77, 'duration': 14373, 'x': 1685656940}, {'y': 0.73, 'duration': 7189, 'x': 1685671313}, {'y': 0.75, 'duration': 10780, 'x': 1685678502}, {'y': 0.76, 'duration': 10778, 'x': 1685689282}, {'y': 0.78, 'duration': 10779, 'x': 1685700060}, {'y': 0.8, 'duration': 10782, 'x': 1685710839}, {'y': 0.795, 'duration': 10822, 'x': 1685721621}, {'y': 0.82, 'duration': 10780, 'x': 1685732443}, {'y': 0.77, 'duration': 14675, 'x': 1685743223}, {'y': 0.73, 'duration': 7186, 'x': 1685757

KeyboardInterrupt: 